# Experiment 03

# MOREPOC Reflectance Extraction

## Objective

The objective of this notebook is to extract harmonized Landsat surface reflectance values for every Indian MOREPOC sampling observation.

The notebook uses the scene-assigned dataset generated in Experiment 02 and extracts six spectral bands from the corresponding Landsat image.

The extracted reflectance values will be combined with field-measured particulate organic carbon (POC) observations to produce the final machine-learning dataset for Random Forest model development.

---

### Input

Indian_MOREPOC_SceneAssigned.csv

---

### Output

Indian_MOREPOC_Reflectance.csv

---

### Spectral Bands

- Blue
- Green
- Red
- NIR
- SWIR1
- SWIR2

In [1]:
# ============================================================
# Section 2 : Import Libraries
# ============================================================

import ee

import geemap

import pandas as pd

import numpy as np

from datetime import datetime

pd.set_option("display.max_columns", None)

print("="*70)
print("Libraries Imported Successfully")
print("="*70)

Libraries Imported Successfully


# Section 3 : Initialize Google Earth Engine

## Objective

This section initializes the Google Earth Engine Python API and creates an interactive map using geemap.

A successful initialization confirms that the Earth Engine project is authenticated and ready for image processing.

The interactive map also provides a quick visual reference for subsequent analyses.

In [2]:
# ============================================================
# Section 3.1 : Initialize Google Earth Engine
# ============================================================

try:

    ee.Initialize(project="oc-flux")

    print("="*70)
    print("Google Earth Engine Initialized Successfully")
    print("="*70)

except Exception as e:

    print("Earth Engine Initialization Failed")
    print(e)

Google Earth Engine Initialized Successfully


In [3]:
# ============================================================
# Section 3.2 : Interactive Map
# ============================================================

Map = geemap.Map()

Map.setCenter(
    82.8,
    22.5,
    5
)

Map

Map(center=[22.5, 82.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [4]:
# ============================================================
# Section 3.3 : Earth Engine Verification
# ============================================================

print("="*70)
print("Earth Engine Version Check")
print("="*70)

print(type(ee.Image(1)))

print()

print("="*70)
print("Geometry Test")
print("="*70)

test_point = ee.Geometry.Point([77.2, 28.6])

print(test_point.getInfo())

Earth Engine Version Check
<class 'ee.image.Image'>

Geometry Test
{'type': 'Point', 'coordinates': [77.2, 28.6]}


# Section 4 : Load Scene Assigned Dataset

## Objective

This section imports the Landsat scene-assigned dataset generated in Experiment 02.

The dataset contains:

- Sampling information
- Assigned Landsat Scene ID
- Landsat Sensor
- Acquisition Date
- Cloud Cover
- Temporal Difference

This dataset will be converted into an Earth Engine FeatureCollection for reflectance extraction.

In [5]:
# ============================================================
# Section 4.1 : Load Scene Assigned Dataset
# ============================================================

assigned_df = pd.read_csv(

    "Indian_MOREPOC_SceneAssigned.csv"

)

print("="*70)
print("Scene Assigned Dataset Loaded")
print("="*70)

print("Shape :", assigned_df.shape)

display(assigned_df.head())

Scene Assigned Dataset Loaded
Shape : (27, 13)


,cloud_cover,conc_poc,country,date_difference,image_count,riv_id,sample_date,sample_id,scene_date,scene_id,sensor,lon,lat
0,51,2.6,India,31.841038,3,Brahmaputra,1997-03-15,IND_0006,1997-02-11,1_1_1_LT05_137042_19970211,LANDSAT_5,91.74,26.19
1,51,0.1,India,31.841038,3,Brahmaputra,1997-03-15,IND_0007,1997-02-11,1_1_1_LT05_137042_19970211,LANDSAT_5,91.74,26.19
2,34,3.9,India,0.837279,19,Brahmaputra,1999-10-25,IND_0001,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
3,34,5.4,India,0.837279,19,Brahmaputra,1999-10-25,IND_0002,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19
4,34,5.9,India,0.837279,19,Brahmaputra,1999-10-25,IND_0003,1999-10-24,1_1_1_LT05_136042_19991024,LANDSAT_5,91.74,26.19


In [6]:
# ============================================================
# Section 4.2 : Dataset Information
# ============================================================

print("="*70)
print("Dataset Information")
print("="*70)

assigned_df.info()

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cloud_cover      27 non-null     int64  
 1   conc_poc         27 non-null     float64
 2   country          27 non-null     str    
 3   date_difference  27 non-null     float64
 4   image_count      27 non-null     int64  
 5   riv_id           27 non-null     str    
 6   sample_date      27 non-null     str    
 7   sample_id        27 non-null     str    
 8   scene_date       27 non-null     str    
 9   scene_id         27 non-null     str    
 10  sensor           27 non-null     str    
 11  lon              27 non-null     float64
 12  lat              27 non-null     float64
dtypes: float64(4), int64(2), str(7)
memory usage: 2.9 KB


In [7]:
# ============================================================
# Section 4.3 : Missing Values
# ============================================================

print("="*70)
print("Missing Values")
print("="*70)

print(

assigned_df.isna().sum()

)

Missing Values
cloud_cover        0
conc_poc           0
country            0
date_difference    0
image_count        0
riv_id             0
sample_date        0
sample_id          0
scene_date         0
scene_id           0
sensor             0
lon                0
lat                0
dtype: int64


In [8]:
# ============================================================
# Section 4.4 : Sensor Summary
# ============================================================

print("="*70)
print("Sensor Distribution")
print("="*70)

print(

assigned_df["sensor"].value_counts()

)

Sensor Distribution
sensor
LANDSAT_5    14
LANDSAT_7    13
Name: count, dtype: int64


In [9]:
# ============================================================
# Section 4.5 : Temporal Difference
# ============================================================

print("="*70)
print("Date Difference Statistics")
print("="*70)

print(

assigned_df["date_difference"].describe()

)

Date Difference Statistics
count    27.000000
mean      4.646666
std       7.958572
min       0.192924
25%       0.837279
50%       2.807076
75%       4.178295
max      31.841038
Name: date_difference, dtype: float64


### Interpretation

The scene-assigned dataset has been successfully imported.

Each sampling observation is linked to a Landsat scene through:

- Scene ID
- Sensor
- Scene Date
- Cloud Cover
- Temporal Difference

This verified dataset will now be converted into an Earth Engine FeatureCollection for automated reflectance extraction.

# Section 5 : Rebuild Earth Engine FeatureCollection

## Objective

The scene-assigned dataset is converted back into an Earth Engine FeatureCollection.

Each observation contains:

- Sampling coordinates
- Sample ID
- Sampling date
- Assigned Landsat scene metadata

This FeatureCollection will be used for automated spectral reflectance extraction.

# Section 5 : Create Earth Engine FeatureCollection

## Objective

Convert the scene-assigned dataset into an Earth Engine FeatureCollection.

Each feature contains:

- Sample location (Latitude & Longitude)
- Sample ID
- Sample Date
- Assigned Landsat Scene
- Sensor
- POC Concentration

This FeatureCollection will be used for automated reflectance extraction.

In [10]:
# ============================================================
# Section 5.1 : DataFrame → FeatureCollection
# ============================================================

features = []

for _, row in assigned_df.iterrows():

    feature = ee.Feature(

        ee.Geometry.Point(
            [row["lon"], row["lat"]]
        ),

        {
            "sample_id": row["sample_id"],
            "sample_date": row["sample_date"],
            "scene_date": row["scene_date"],
            "scene_id": row["scene_id"],
            "sensor": row["sensor"],
            "conc_poc": row["conc_poc"],
            "country": row["country"],
            "riv_id": row["riv_id"],
            "cloud_cover": row["cloud_cover"],
            "date_difference": row["date_difference"]
        }

    )

    features.append(feature)

assigned_fc = ee.FeatureCollection(features)

print("="*70)
print("FeatureCollection Created")
print("="*70)

print("Number of Features :", assigned_fc.size().getInfo())

FeatureCollection Created
Number of Features : 27


In [11]:
# ============================================================
# Section 5.2 : Inspect First Feature
# ============================================================

first_feature = assigned_fc.first()

print("="*70)
print("First Feature")
print("="*70)

print(first_feature.getInfo())

First Feature
{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [91.74, 26.19]}, 'id': '0', 'properties': {'cloud_cover': 51, 'conc_poc': 2.6, 'country': 'India', 'date_difference': 31.841038460648143, 'riv_id': 'Brahmaputra ', 'sample_date': '1997-03-15', 'sample_id': 'IND_0006', 'scene_date': '1997-02-11', 'scene_id': '1_1_1_LT05_137042_19970211', 'sensor': 'LANDSAT_5'}}


# Section 6 : Load Landsat Collections

## Objective

Load the Landsat Surface Reflectance collections that correspond to the assigned scenes.

Only Collection 2 Level 2 products are used.

Collections:

- Landsat 5
- Landsat 7
- Landsat 8
- Landsat 9

These collections will later be queried using the assigned scene date and sampling location.

In [12]:
# ============================================================
# Section 6.1 : Load Landsat Collections
# ============================================================

LS5 = ee.ImageCollection(
    "LANDSAT/LT05/C02/T1_L2"
)

LS7 = ee.ImageCollection(
    "LANDSAT/LE07/C02/T1_L2"
)

LS8 = ee.ImageCollection(
    "LANDSAT/LC08/C02/T1_L2"
)

LS9 = ee.ImageCollection(
    "LANDSAT/LC09/C02/T1_L2"
)

print("="*70)
print("Landsat Collections Loaded")
print("="*70)

Landsat Collections Loaded


In [13]:
# ============================================================
# Section 6.2 : Collection Sizes
# ============================================================

print("="*70)
print("Collection Sizes")
print("="*70)

print("LS5 :", LS5.size().getInfo())
print("LS7 :", LS7.size().getInfo())
print("LS8 :", LS8.size().getInfo())
print("LS9 :", LS9.size().getInfo())

Collection Sizes
LS5 : 1876269
LS7 : 2580169
LS8 : 2048380
LS9 : 729070


In [14]:
# ============================================================
# Section 6.3 : First Image Check
# ============================================================

print("="*70)
print("Landsat 5 First Image")
print("="*70)

first = LS5.first()

print(first.get("SPACECRAFT_ID").getInfo())

print(first.bandNames().getInfo())

Landsat 5 First Image
LANDSAT_5
['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'SR_ATMOS_OPACITY', 'SR_CLOUD_QA', 'ST_B6', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


# Section 7 : Landsat Image Pre-processing

## Objective

Before extracting spectral reflectance values, Landsat imagery must undergo a series of preprocessing operations to ensure consistency across different sensors and improve the reliability of the extracted data.

The preprocessing workflow consists of three major steps:

- Cloud and cloud-shadow masking using the QA_PIXEL band.
- Conversion of Collection 2 Level-2 digital numbers into surface reflectance values using official USGS scaling factors.
- Harmonization of Landsat 5, Landsat 7, Landsat 8, and Landsat 9 spectral bands into a common band naming convention.

These preprocessing steps produce a standardized six-band reflectance dataset suitable for subsequent spectral analysis and machine-learning model development.

In [15]:
# ============================================================
# Section 7.1 : Relaxed Cloud Mask
# ============================================================

def mask_clouds(image):

    qa = image.select("QA_PIXEL")

    mask = qa.bitwiseAnd(1 << 3).eq(0)

    return image.updateMask(mask)

In [16]:
# ============================================================
# Section 7.2 : Harmonize Landsat Bands
# ============================================================

def harmonize(image):

    spacecraft = ee.String(image.get("SPACECRAFT_ID"))

    is_old = spacecraft.compareTo("LANDSAT_5").eq(0).Or(
        spacecraft.compareTo("LANDSAT_7").eq(0)
    )

    old = image.select(
        ["SR_B1","SR_B2","SR_B3","SR_B4","SR_B5","SR_B7"],
        ["Blue","Green","Red","NIR","SWIR1","SWIR2"]
    )

    new = image.select(
        ["SR_B2","SR_B3","SR_B4","SR_B5","SR_B6","SR_B7"],
        ["Blue","Green","Red","NIR","SWIR1","SWIR2"]
    )

    return ee.Image(
        ee.Algorithms.If(
            is_old,
            old,
            new
        )
    )

In [17]:
# ============================================================
# Section 7.3 : Surface Reflectance Scaling
# ============================================================

def apply_scale(image):

    optical = image.select(
        "SR_B.*"
    ).multiply(
        0.0000275
    ).add(
        -0.2
    )

    return image.addBands(
        optical,
        overwrite=True
    )

In [18]:
# ============================================================
# Section 7.4 : Complete Image Preparation
# ============================================================

def prepare_image(image):

    # Do NOT mask clouds for training samples

    image = apply_scale(image)

    image = harmonize(image)

    return image

In [19]:
# ============================================================
# Section 7.5 : Test Image Pipeline
# ============================================================

test = prepare_image(
    LS5.first()
)

print("="*70)
print("Prepared Image Bands")
print("="*70)

print(
    test.bandNames().getInfo()
)

Prepared Image Bands
['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']


## Interpretation

The image preprocessing pipeline has been successfully established.

Three important preprocessing operations are performed before reflectance extraction:

1. **Cloud and Shadow Masking**
   - Removes pixels affected by clouds and cloud shadows using the QA_PIXEL band.

2. **Surface Reflectance Scaling**
   - Converts the integer digital numbers of Landsat Collection 2 Level-2 products into physically meaningful surface reflectance values using the official USGS scale factors.

3. **Band Harmonization**
   - Standardizes Landsat 5, Landsat 7, Landsat 8, and Landsat 9 spectral bands into a common band naming scheme:
     - Blue
     - Green
     - Red
     - NIR
     - SWIR1
     - SWIR2

Band harmonization ensures that all Landsat sensors produce a consistent spectral dataset, allowing subsequent machine-learning models to use a unified predictor set regardless of sensor type.

The preprocessing pipeline has been verified successfully and is now ready for automated reflectance extraction.

# Section 8 : Landsat Image Selection Pipeline

## Objective

The objective of this section is to retrieve the Landsat image associated with each sampling observation.

Instead of reconstructing images from stored scene identifiers, images are retrieved directly from the appropriate Landsat archive using:

- Landsat Sensor
- Acquisition Date
- Sampling Location

For each observation, a single Landsat Collection 2 Level-2 image is selected and preprocessed before spectral reflectance extraction.

This approach is computationally robust, minimizes Earth Engine object-casting errors, and follows standard remote sensing workflows.

In [20]:
# ============================================================
# Section 8.1 : Select Landsat Collection
# ============================================================

def get_collection(sensor):

    collections = {

        "LANDSAT_5": LS5,
        "LANDSAT_7": LS7,
        "LANDSAT_8": LS8,
        "LANDSAT_9": LS9

    }

    return collections.get(sensor)

In [21]:
# ============================================================
# Section 8.2 : Retrieve Assigned Image
# ============================================================

def get_image(feature):

    sensor = feature.get("sensor")

    collection = get_collection(sensor.getInfo())

    scene_date = ee.Date(feature.get("scene_date"))

    point = feature.geometry()

    image = (

        collection

        .filterBounds(point)

        .filterDate(
            scene_date.advance(-1, "day"),
            scene_date.advance(1, "day")
        )

        .sort("CLOUD_COVER")

        .first()

    )

    return ee.Image(image)

In [22]:
# ============================================================
# Section 8.3 : Test Image Retrieval
# ============================================================

sample = assigned_fc.first()

image = get_image(sample)

print("=" * 70)
print("Retrieved Image")
print("=" * 70)

print(image.get("LANDSAT_PRODUCT_ID").getInfo())

Retrieved Image
LT05_L2SP_137042_19970211_20200911_02_T1


In [23]:
# ============================================================
# Section 8.4 : Display Selected Image
# ============================================================

Map = geemap.Map()

Map.centerObject(sample.geometry(), 9)

Map.addLayer(

    image,

    {

        "bands": ["SR_B3", "SR_B2", "SR_B1"],

        "min": 7000,

        "max": 18000

    },

    "Original Image"

)

Map

Map(center=[26.19, 91.74], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

## Interpretation

The image selection workflow successfully retrieves the Landsat Collection 2 Level-2 image corresponding to each sampling observation.

Image selection is based on the assigned sensor, acquisition date, and sampling location rather than relying solely on stored scene identifiers.

This approach improves robustness, reduces dependency on manually reconstructed scene identifiers, and ensures compatibility with Earth Engine image collections.

The selected image will be passed through the preprocessing pipeline before spectral reflectance values are extracted in the following section.

# Section 9 : Server-side Reflectance Extraction

## Objective

This section extracts Landsat surface reflectance values using a fully server-side workflow.

To avoid Earth Engine client/server conflicts, samples are processed separately for each Landsat sensor.

Workflow:

1. Split samples according to sensor.
2. Load the corresponding Landsat archive.
3. Retrieve the assigned acquisition date.
4. Apply preprocessing.
5. Extract reflectance.
6. Merge all results.

This workflow is scalable and follows recommended Earth Engine practices.

In [24]:
# ============================================================
# Section 9.1 : Split Samples by Sensor
# ============================================================

ls5_df = assigned_df[
    assigned_df["sensor"] == "LANDSAT_5"
].copy()

ls7_df = assigned_df[
    assigned_df["sensor"] == "LANDSAT_7"
].copy()

ls8_df = assigned_df[
    assigned_df["sensor"] == "LANDSAT_8"
].copy()

ls9_df = assigned_df[
    assigned_df["sensor"] == "LANDSAT_9"
].copy()

print("="*70)
print("Samples per Sensor")
print("="*70)

print("LS5 :", len(ls5_df))
print("LS7 :", len(ls7_df))
print("LS8 :", len(ls8_df))
print("LS9 :", len(ls9_df))

Samples per Sensor
LS5 : 14
LS7 : 13
LS8 : 0
LS9 : 0


In [25]:
# ============================================================
# Section 9.2 : Extract Reflectance
# ============================================================

def extract_dataset(df, collection):

    results = []

    for _, row in df.iterrows():

        point = ee.Geometry.Point(
            [row["lon"], row["lat"]]
        )

        scene_date = ee.Date(row["scene_date"])

        image = (

            collection

            .filterBounds(point)

            .filterDate(
                scene_date.advance(-1, "day"),
                scene_date.advance(1, "day")
            )

            .sort("CLOUD_COVER")

            .first()

        )

        image = prepare_image(image)

        values = image.reduceRegion(

            reducer=ee.Reducer.first(),

            geometry=point,

            scale=30,

            bestEffort=True

        ).getInfo()

        values["sample_id"] = row["sample_id"]

        values["conc_poc"] = row["conc_poc"]

        values["sensor"] = row["sensor"]

        values["scene_date"] = row["scene_date"]

        results.append(values)

    return pd.DataFrame(results)

In [26]:
# ============================================================
# Section 9.3 : Landsat 5
# ============================================================

ls5_reflectance = extract_dataset(
    ls5_df,
    LS5
)

print(ls5_reflectance.shape)

display(ls5_reflectance.head())

(14, 10)


,Blue,Green,NIR,Red,SWIR1,SWIR2,sample_id,conc_poc,sensor,scene_date
0,0.280342,0.272560,0.286778,0.278417,0.224353,0.192783,IND_0006,2.6,LANDSAT_5,1997-02-11
1,0.280342,0.272560,0.286778,0.278417,0.224353,0.192783,IND_0007,0.1,LANDSAT_5,1997-02-11
2,0.475702,0.496107,0.498115,0.494127,0.445947,0.372550,IND_0001,3.9,LANDSAT_5,1999-10-24
3,0.475702,0.496107,0.498115,0.494127,0.445947,0.372550,IND_0002,5.4,LANDSAT_5,1999-10-24
4,0.475702,0.496107,0.498115,0.494127,0.445947,0.372550,IND_0003,5.9,LANDSAT_5,1999-10-24


In [27]:
# ============================================================
# Section 9.4 : Landsat 7
# ============================================================

ls7_reflectance = extract_dataset(
    ls7_df,
    LS7
)

print(ls7_reflectance.shape)

display(ls7_reflectance.head())

(13, 10)


,Blue,Green,NIR,Red,SWIR1,SWIR2,sample_id,conc_poc,sensor,scene_date
0,0.063698,0.090373,0.130082,0.090070,0.060783,0.032265,IND_0012,7.1,LANDSAT_7,2001-08-05
1,0.116415,0.162257,0.177218,0.197100,0.056107,0.049260,IND_0013,6.5,LANDSAT_7,2001-08-05
2,0.116415,0.162257,0.177218,0.197100,0.056107,0.049260,IND_0014,5.8,LANDSAT_7,2001-08-05
3,0.116415,0.162257,0.177218,0.197100,0.056107,0.049260,IND_0015,6.7,LANDSAT_7,2001-08-05
4,0.062982,0.087952,0.309328,0.084323,0.185440,0.107478,IND_0016,5.9,LANDSAT_7,2001-08-05


In [28]:
# ============================================================
# Section 9.5 : Debug Landsat-5 Sample
# ============================================================

sample = ls5_df.iloc[0]

print(sample)

point = ee.Geometry.Point([sample["lon"], sample["lat"]])

scene_date = ee.Date(sample["scene_date"])

image = (
    LS5
    .filterBounds(point)
    .filterDate(
        scene_date.advance(-1, "day"),
        scene_date.advance(1, "day")
    )
    .first()
)

print("=" * 70)
print("Image Exists")
print("=" * 70)

print(image.getInfo() is not None)

print("=" * 70)
print("Band Names")
print("=" * 70)

print(image.bandNames().getInfo())

cloud_cover                                51
conc_poc                                  2.6
country                                 India
date_difference                     31.841038
image_count                                 3
riv_id                           Brahmaputra 
sample_date                        1997-03-15
sample_id                            IND_0006
scene_date                         1997-02-11
scene_id           1_1_1_LT05_137042_19970211
sensor                              LANDSAT_5
lon                                     91.74
lat                                     26.19
Name: 0, dtype: object
Image Exists
True
Band Names
['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'SR_ATMOS_OPACITY', 'SR_CLOUD_QA', 'ST_B6', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


In [29]:
Map = geemap.Map()

Map.centerObject(point, 9)

Map.addLayer(
    image,
    {
        "bands": ["SR_B3", "SR_B2", "SR_B1"],
        "min": 7000,
        "max": 18000
    },
    "Raw Image"
)

Map.addLayer(point, {"color": "red"}, "Sample")

Map

Map(center=[26.19, 91.74], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [30]:
raw = image.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=30
)

print(raw.getInfo())

{'QA_PIXEL': 5896, 'QA_RADSAT': 0, 'SR_ATMOS_OPACITY': 270, 'SR_B1': 17467, 'SR_B2': 17184, 'SR_B3': 17397, 'SR_B4': 17701, 'SR_B5': 15431, 'SR_B7': 14283, 'SR_CLOUD_QA': 8, 'ST_ATRAN': 8145, 'ST_B6': 38466, 'ST_CDIST': 0, 'ST_DRAD': 655, 'ST_EMIS': 9880, 'ST_EMSD': 0, 'ST_QA': 648, 'ST_TRAD': 6886, 'ST_URAD': 1325}


In [31]:
scaled = apply_scale(image)

print(
    scaled.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30
    ).getInfo()
)

{'QA_PIXEL': 5896, 'QA_RADSAT': 0, 'SR_ATMOS_OPACITY': 270, 'SR_B1': 0.2803425, 'SR_B2': 0.27256, 'SR_B3': 0.2784175, 'SR_B4': 0.2867775, 'SR_B5': 0.2243525, 'SR_B7': 0.19278250000000002, 'SR_CLOUD_QA': 8, 'ST_ATRAN': 8145, 'ST_B6': 38466, 'ST_CDIST': 0, 'ST_DRAD': 655, 'ST_EMIS': 9880, 'ST_EMSD': 0, 'ST_QA': 648, 'ST_TRAD': 6886, 'ST_URAD': 1325}


In [32]:
harm = harmonize(image)

print(
    harm.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30
    ).getInfo()
)

{'Blue': 17467, 'Green': 17184, 'NIR': 17701, 'Red': 17397, 'SWIR1': 15431, 'SWIR2': 14283}


In [33]:
prep = prepare_image(image)

print(
    prep.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30
    ).getInfo()
)

{'Blue': 0.2803425, 'Green': 0.27256, 'NIR': 0.2867775, 'Red': 0.2784175, 'SWIR1': 0.2243525, 'SWIR2': 0.19278250000000002}


In [34]:
qa = image.select("QA_PIXEL")

print("QA_PIXEL:", qa.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=30
).getInfo())

QA_PIXEL: {'QA_PIXEL': 5896}


## Interpretation

The reflectance extraction workflow processes each Landsat sensor independently, eliminating client-side operations inside Earth Engine mapping functions.

Each observation is linked to its assigned Landsat image, preprocessed, and sampled at the field location.

The resulting datasets contain harmonized surface reflectance values for the six predictor bands required for particulate organic carbon modelling.

Processing sensors separately improves workflow robustness and simplifies debugging while maintaining compatibility with large datasets.